# Step 19: Advanced: Multi-Agent Evaluation and Trace Analysis

        _Instructor Solution Notebook_  
        Level: **Advanced**  
        Estimated time: **110 minutes**

        ## Learning Objectives
        - [ ] Build a lightweight evaluation loop for agent outputs.
- [ ] Compare agent behavior against explicit rubric criteria.
- [ ] Capture simple traces and review failure patterns.
- [ ] Use evaluation results to guide prompt and orchestration changes.

        ## Prerequisites
        - Core Steps 12-15 completed.
- Comfort reviewing prompt and orchestration outputs critically.


## Table of Contents

1. Setup & Imports
2. Part 1: Introduction
3. Part 2: Core Implementation
4. Part 3: Hands-On Exercises
5. Part 4: Solutions & Discussion
6. Part 5: Best Practices & Tips
7. Reflection & Next Experiments
8. Summary & Key Takeaways
9. Additional Resources


## Setup & Imports

Supplemental notebooks assume the core helpers are available and that you have already worked
through the matching core lessons. The import cell intentionally keeps the same bootstrap shape
as the core course so you can focus on the deeper pattern rather than environment setup.


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "helpers").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from helpers import (
    LocalTfidfVectorStore,
    SQLiteConversationMemory,
    assert_minimum_python,
    chunk_documents,
    create_agent,
    create_chat_client,
    describe_response,
    load_settings,
    load_text_documents,
    print_banner,
    print_json,
    resolve_notebook_root,
    summarize_session,
    validate_provider_configuration,
)


## Part 1: Introduction

Advanced agent work is iterative. This notebook makes evaluation and trace review part of the development loop instead of something added after deployment.

This notebook is intentionally deeper than the core path. Expect more design discussion,
more implementation sections, and more open-ended exercises.


## Part 2: Core Implementation

### Create a tiny evaluation dataset


In [ ]:
eval_cases = [
    {
        "query": "Explain why tool descriptions matter.",
        "expected_keywords": ["tool", "description", "model", "selection"],
    },
    {
        "query": "What is the difference between branching and fan-out in workflows?",
        "expected_keywords": ["branch", "fan-out", "workflow", "route"],
    },
]

judge = create_agent(
    name="JudgeAgent",
    instructions="Score answers from 0 to 5 for clarity, correctness, and usefulness. Return a short justification.",
)
candidate = create_agent(
    name="CandidateAgent",
    instructions="Answer course questions clearly for developers learning agent systems.",
)


## Part 2: Core Implementation

### Generate answers and record traces


In [ ]:
traces = []
for case in eval_cases:
    answer = await candidate.run(case["query"])
    trace = {
        "query": case["query"],
        "answer": answer.text,
        "keyword_hits": [kw for kw in case["expected_keywords"] if kw.lower() in answer.text.lower()],
    }
    traces.append(trace)

print_json(traces, title="Candidate traces")


## Part 2: Core Implementation

### Ask a judge model for qualitative review


In [ ]:
judged = []
for trace in traces:
    review = await judge.run(
        f"""Question: {trace['query']}
        Answer: {trace['answer']}
        Score the answer from 0 to 5 and explain why."""
    )
    judged.append({**trace, "judge_review": review.text})

print_json(judged, title="Judged traces")


## Part 2: Core Implementation

### Summarize recurring failure modes


In [ ]:
def trace_summary(items: list[dict]) -> dict:
    return {
        "case_count": len(items),
        "average_keyword_hits": sum(len(item["keyword_hits"]) for item in items) / len(items),
        "common_risk": "Answers may sound polished even when they omit required vocabulary.",
    }

print_json(trace_summary(judged), title="Evaluation summary")


## Part 3: Hands-On Exercises

### Exercise 1
Add a new evaluation case that targets orchestration tradeoffs rather than definitions.

### Exercise 2
Write a helper that flags traces with fewer than two keyword hits as risky outputs.

This solution notebook includes completed code for both guided exercises.


In [ ]:
eval_cases.append(
    {
        "query": "When should you prefer sequential orchestration over concurrent orchestration?",
        "expected_keywords": ["sequential", "concurrent", "dependency", "parallel"],
    }
)
print_json(eval_cases[-1], title="Exercise 1 solution")


In [ ]:
def flag_risky_trace(trace: dict) -> bool:
    return len(trace["keyword_hits"]) < 2

print_json(
    [flag_risky_trace(trace) for trace in traces],
    title="Exercise 2 solution",
)


## Part 4: Solutions & Discussion

Use this section to compare your implementation with one clear working approach. The goal is not
perfect matching output; the goal is a sound pattern that is easy to explain, debug, and extend.


In [ ]:
eval_cases.append(
    {
        "query": "When should you prefer sequential orchestration over concurrent orchestration?",
        "expected_keywords": ["sequential", "concurrent", "dependency", "parallel"],
    }
)
print_json(eval_cases[-1], title="Exercise 1 solution")


In [ ]:
def flag_risky_trace(trace: dict) -> bool:
    return len(trace["keyword_hits"]) < 2

print_json(
    [flag_risky_trace(trace) for trace in traces],
    title="Exercise 2 solution",
)


## Part 5: Best Practices & Tips

        - Keep evaluation cases small enough to inspect manually at first.
- Combine quantitative checks with rubric-style reviews.
- Store traces in a form that makes failure analysis easy later.
- Use evaluation to refine prompts, tool contracts, or orchestration shape.


## Reflection & Next Experiments

- Which part of `Advanced: Multi-Agent Evaluation and Trace Analysis` felt like the biggest jump from the core course?
- What would you keep deterministic before turning this into a live production feature?
- Where would you add tests, traces, or operator controls before shipping this pattern?


## Summary & Key Takeaways

You turned evaluation and trace review into a repeatable engineering loop instead of a vague quality goal.


## Additional Resources

        - `core notebook: 15_production_features.ipynb`
- `docs/references.md`
